# GPSR Command Extraction — QLoRA Fine-tune (Unsloth)

Trains **Qwen3-4B-Instruct-2507** (Unsloth Dynamic 4-bit quant) to map a GPSR-style natural language command -> structured JSON action/slot list, matching the schema in `oldGPSR_testset.jsonl`.

**Runtime:** Colab, GPU (T4 works; the 4B model uses more VRAM than the old 3B baseline, so batch size is tuned down accordingly — see Section 5). `Runtime > Change runtime type > GPU`

**Note:** this uses the **non-thinking** Instruct-2507 checkpoint, not `qwen3-4b-thinking-2507`. Chat-template calls explicitly pass `enable_thinking=False` throughout so no `<think>` reasoning tokens get generated — you don't want that latency/token overhead for real-time structured extraction.

## 1. Install deps

In [ ]:
!pip install -q unsloth
!pip install -q --upgrade --no-deps "git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes

## 2. Load base model (4-bit)
Using Unsloth's Dynamic 4-bit quant of Qwen3-4B-Instruct-2507 — calibrated quantization (keeps sensitive layers higher precision) rather than naive bnb, so quality holds up better than a plain 4-bit cast. To go back to comparing against the old baseline, just swap `MODEL_NAME` back to `"unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"`.

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507-unsloth-bnb-4bit"  # Unsloth Dynamic 4-bit quant
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,          # auto-detect (bf16 on modern GPUs, fp16 on T4)
    load_in_4bit = True,
)

## 3. Attach LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA rank — 16 is a solid default for structured extraction; same value works fine at 4B
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,            # 0 is optimized in Unsloth's fused kernels
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # long-context friendly, saves VRAM
    random_state = 3407,
)

## 4. Load + format your dataset

This matches the **exact format of `oldGPSR_testset.jsonl`** (the team's existing test set) so train and eval never need reshaping — each line is:
```json
{"messages": [
  {"role": "user", "content": "Parse this robot command:\n<command text>"},
  {"role": "assistant", "content": "[{\"sequence\":1,\"action\":\"navigateToLocation\",\"room\":\"study\"}]"}
]}
```
The assistant content is a **JSON string** (a list of step objects, each with `sequence`, `action`, and action-specific slots) — matching `gpsr_schema.json`, the schema extracted from the team's existing eval set.

**Critical: train/test must not overlap.** Upload your generated `gpsr_train.jsonl` here — it must be built from entity pools (locations/objects/names) and action+slot combinations that are disjoint from anything in `oldGPSR_testset.jsonl`, not just superficially different sentences. See the labeling-spec discussion for how the split was designed.

In [ ]:
from datasets import load_dataset
import json

# Option A: upload gpsr_train.jsonl via Colab's file panel, then:
raw_dataset = load_dataset("json", data_files = "gpsr_train.jsonl", split = "train")

# Option B: mount Drive instead
# from google.colab import drive
# drive.mount('/content/drive')
# raw_dataset = load_dataset("json", data_files = "/content/drive/MyDrive/gpsr/gpsr_train.jsonl", split = "train")

def format_example(ex):
    # ex["messages"] is already [{role, content}, ...] — apply the model's chat template directly.
    # enable_thinking=False: Qwen3's template supports a thinking mode: keep it off so no <think> block
    # gets trained into the target output (this is a fixed-schema extraction task, not a reasoning task).
    text = tokenizer.apply_chat_template(
        ex["messages"], tokenize=False, add_generation_prompt=False, enable_thinking=False,
    )
    return {"text": text}

dataset = raw_dataset.map(format_example, remove_columns=raw_dataset.column_names)

# quick sanity check
print(dataset[0]["text"][:800])

## 5. Train

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    packing = False,   # keep False for structured I/O tasks — packing can blur example boundaries
    args = SFTConfig(
        per_device_train_batch_size = 2,   # lower than the 3B baseline (was 4) — 4B model needs more VRAM headroom on a T4
        gradient_accumulation_steps = 8,   # keeps effective batch size at 16, same as before
        warmup_steps = 20,
        num_train_epochs = 3,              # 2-3 is usually enough before overfitting on structured extraction
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "epoch",
        report_to = "none",
    ),
)

trainer_stats = trainer.train()

## 6. Eval against `oldGPSR_testset.jsonl`

Runs inference on every example in the team's existing eval set and scores it two ways, matching the old design mentioned in the project brief:
- **Full score**: 1 point only if `action` AND every slot value match exactly (case-insensitive) — 0 if anything is missing or wrong, even one slot.
- **Action accuracy**: did the model at least predict the right `action` label, regardless of slots?

Both are also broken out **per-step** (single action) and **per-command** (all steps in a multi-step command must be fully correct, in order, for the command-level score) so you can see where multi-step commands fall apart vs single-step ones.

This gives you a baseline to compare your fine-tuned model against — you can re-point `TEST_PATH` at the standardized test set once kavinsky's version is ready, and re-run this cell unchanged.

In [ ]:
import json

TEST_PATH = "oldGPSR_testset.jsonl"   # upload this file to Colab's file panel first

FastLanguageModel.for_inference(model)

def run_raw(user_content):
    messages = [{"role": "user", "content": user_content}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", enable_thinking=False,
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=384, temperature=0.1, do_sample=False)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

def normalize(v):
    # case-insensitive, whitespace-trimmed compare; leaves numbers/bools alone
    return v.strip().lower() if isinstance(v, str) else v

def step_matches(pred_step, gold_step):
    """Full match: action + every gold slot (excluding 'sequence') must match exactly.
    Extra keys in pred beyond gold are ignored (mirrors the old 'must have everything, nothing wrong' rule
    focused on required fields; tighten to `pred_step.keys() == gold_step.keys()` if you want to
    also penalize hallucinated extra slots)."""
    if not isinstance(pred_step, dict):
        return False
    if normalize(pred_step.get("action")) != normalize(gold_step.get("action")):
        return False
    for k, v in gold_step.items():
        if k == "sequence":
            continue
        if normalize(pred_step.get(k)) != normalize(v):
            return False
    return True

results = []
with open(TEST_PATH, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        user_msg = next(m["content"] for m in obj["messages"] if m["role"] == "user")
        gold_raw = next(m["content"] for m in obj["messages"] if m["role"] == "assistant")
        gold_steps = json.loads(gold_raw)

        pred_raw = run_raw(user_msg)
        try:
            cleaned = pred_raw.strip()
            # defensive: strip a stray <think>...</think> block if the template flag gets ignored
            if "<think>" in cleaned and "</think>" in cleaned:
                cleaned = cleaned.split("</think>", 1)[1].strip()
            # strip stray markdown fences if the model adds them despite instructions
            cleaned = cleaned.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            pred_steps = json.loads(cleaned)
            if not isinstance(pred_steps, list):
                pred_steps = []
        except json.JSONDecodeError:
            pred_steps = []

        # per-step scoring, aligned by sequence order
        n_steps = len(gold_steps)
        step_hits = 0
        action_hits = 0
        for i, gold_step in enumerate(gold_steps):
            pred_step = pred_steps[i] if i < len(pred_steps) else {}
            if step_matches(pred_step, gold_step):
                step_hits += 1
            if isinstance(pred_step, dict) and normalize(pred_step.get("action")) == normalize(gold_step.get("action")):
                action_hits += 1

        command_full_score = 1 if (step_hits == n_steps and len(pred_steps) == n_steps) else 0

        results.append({
            "line": line_no,
            "command": user_msg,
            "n_steps": n_steps,
            "step_hits": step_hits,
            "action_hits": action_hits,
            "command_full_score": command_full_score,
            "pred_raw": pred_raw,
        })

n = len(results)
total_steps = sum(r["n_steps"] for r in results)
print(f"Commands evaluated: {n}")
print(f"Command-level full score  : {sum(r['command_full_score'] for r in results)}/{n}  "
      f"({100*sum(r['command_full_score'] for r in results)/n:.1f}%)")
print(f"Step-level full-match rate: {sum(r['step_hits'] for r in results)}/{total_steps}  "
      f"({100*sum(r['step_hits'] for r in results)/total_steps:.1f}%)")
print(f"Action accuracy (step)    : {sum(r['action_hits'] for r in results)}/{total_steps}  "
      f"({100*sum(r['action_hits'] for r in results)/total_steps:.1f}%)")

# breakdown by step-count, to see where multi-step commands fall apart
from collections import defaultdict
by_len = defaultdict(list)
for r in results:
    by_len[r["n_steps"]].append(r["command_full_score"])
print("\nFull score by command length:")
for k in sorted(by_len):
    vals = by_len[k]
    print(f"  {k}-step: {sum(vals)}/{len(vals)}  ({100*sum(vals)/len(vals):.1f}%)")

# save full results for later comparison against the old model / future runs
with open("eval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

## 7. Merge LoRA into base + save
This produces a `merged` model directly analogous to the `gpsr-coder3b-multi-best-merged` file you were sent — standalone weights, no adapter needed at inference time. Named `gpsr-qwen3-4b-merged` so it doesn't overwrite/confuse with a Qwen2.5-Coder-3B run's output.

In [ ]:
OUT_NAME = "gpsr-qwen3-4b-merged"

# 16-bit merged (best quality, larger file — use if you'll requantize later)
model.save_pretrained_merged(f"{OUT_NAME}-16bit", tokenizer, save_method="merged_16bit")

# OR 4-bit merged directly (smaller, ready to deploy as-is)
# model.save_pretrained_merged(f"{OUT_NAME}-4bit", tokenizer, save_method="merged_4bit")

# zip it up for download / handing off to teammates, same pattern as the file you received
!zip -r {OUT_NAME}.zip {OUT_NAME}-16bit

## 8. (Optional) GGUF export for llama.cpp / local inference
Useful if the robot's onboard inference stack runs GGUF rather than raw HF transformers.

In [ ]:
# model.save_pretrained_gguf("gpsr-model-gguf", tokenizer, quantization_method="q4_k_m")

## 9. Generate a NEW synthetic test set using the model

**Why not just ask the model to invent commands *and* labels freely?** Because then you have no ground truth — if the model hallucinates a wrong slot, nothing catches it, and you've built an eval set you can't trust.

Instead this does it in the direction that stays verifiable:
1. **Sample the JSON first**, programmatically, from the same action/slot schema and entity pools as `oldGPSR_testset.jsonl` (extracted automatically below — nothing hardcoded).
2. **Ask the model only to phrase the sentence** for that JSON — LLMs are reliable at "turn this structured spec into natural English," much more so than "invent a structured spec from scratch."
3. **Round-trip check**: feed the generated sentence back through your own extraction (`run_raw`) and require it to reproduce the same JSON. If it doesn't, the sentence was ambiguous — throw it out.
4. Save survivors to `gpsr_generated_testset.jsonl` in the exact same `{"messages": [...]}` schema, skipping anything that duplicates an existing old-test-set example.

This guarantees every label in the new test set is actually correct, while the *sentences* are fresh model-generated paraphrases — no manual template writing needed.

In [ ]:
import json, random, re
from collections import defaultdict

OLD_TEST_PATH = "oldGPSR_testset.jsonl"

# ---- 9.1 Extract schema + entity pools from the old test set (fully automatic, nothing hardcoded) ----
action_slots = defaultdict(set)      # action -> set of slot names it uses
entity_pools = defaultdict(set)      # slot name -> set of values seen for it
step_count_dist = []                 # e.g. [1,1,2,3,...] to preserve the old 1/2/3-step mix
existing_step_sets = set()           # to check exact-duplicate steps against the old set

with open(OLD_TEST_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        gold = json.loads(next(m["content"] for m in obj["messages"] if m["role"] == "assistant"))
        step_count_dist.append(len(gold))
        for step in gold:
            act = step["action"]
            slots = {k: v for k, v in step.items() if k not in ("sequence", "action")}
            for k, v in slots.items():
                action_slots[act].add(k)
                entity_pools[k].add(v)
            existing_step_sets.add((act, tuple(sorted(slots.items()))))

actions = list(action_slots.keys())
print(f"Extracted {len(actions)} actions, {len(entity_pools)} slot types.")
print("Step-count distribution to preserve:", {n: step_count_dist.count(n) for n in set(step_count_dist)})

# ---- 9.2 (Optional) add brand-new entity values so the new set isn'''t just old entities recombined ----
# Extend these lists with new names/locations/objects if you want genuinely novel content
# rather than just new combinations of the old entities. Safe to leave empty.
NEW_ENTITIES = {
    "person_name": [],   # e.g. ["Marcus", "Priya"]
    "object": [],        # e.g. ["umbrella", "notebook"]
    "room": [],
}
for slot, extra_vals in NEW_ENTITIES.items():
    entity_pools[slot].update(extra_vals)


In [ ]:
# ---- 9.3 Sample brand-new (action, slots) combinations, disjoint from the old test set ----

def sample_step(action, sequence):
    slots = {}
    for slot_name in action_slots[action]:
        slots[slot_name] = random.choice(list(entity_pools[slot_name]))
    return {"sequence": sequence, "action": action, **slots}

def sample_command(n_steps):
    """Sample n_steps distinct actions and fill their slots, retrying until the
    whole step-set doesn'''t already exist verbatim in the old test set."""
    for _ in range(50):
        chosen_actions = random.sample(actions, k=min(n_steps, len(actions)))
        steps = [sample_step(a, i + 1) for i, a in enumerate(chosen_actions)]
        key_set = {(s["action"], tuple(sorted((k, v) for k, v in s.items() if k not in ("sequence", "action"))))
                   for s in steps}
        if not (key_set & existing_step_sets):
            return steps
    return None  # gave up after 50 tries (pools too small) — caller should skip

N_NEW_EXAMPLES = 150   # how many new test-set rows to attempt
target_lengths = random.choices([1, 2, 3], k=N_NEW_EXAMPLES)  # mirror the old 1/2/3-step mix; adjust weights if needed

candidates = []
for n in target_lengths:
    steps = sample_command(n)
    if steps:
        candidates.append(steps)

print(f"Sampled {len(candidates)} novel action/slot combinations (target was {N_NEW_EXAMPLES}).")


In [ ]:
# ---- 9.4 Use the model to phrase each structured command as natural language ----
# Turning the LoRA adapter OFF for this step: it was fine-tuned to go text -> JSON, and that
# specialization can make its free-form generation clunkier. The base chat model paraphrases better.
# (Skip the `with model.disable_adapter():` line and just call model.generate directly if you'''d
# rather use the fine-tuned model as-is, or if you'''re running this in a fresh session with only the
# base model loaded.)

FastLanguageModel.for_inference(model)

def describe_steps_for_prompt(steps):
    parts = []
    for s in steps:
        slot_str = ", ".join(f"{k}={v}" for k, v in s.items() if k not in ("sequence", "action"))
        parts.append(f"{s['action']}({slot_str})")
    return "; then ".join(parts)

def generate_command_text(steps, temperature=0.9):
    spec = describe_steps_for_prompt(steps)
    prompt = (
        "You write natural, varied English commands for a home service robot competition (GPSR/RoboCup style).\n"
        f"Write ONE single command sentence (it may have multiple clauses) that instructs the robot to do exactly "
        f"this, in this order: {spec}\n"
        "Rules: output ONLY the command sentence, no JSON, no explanation, no quotes. "
        "Vary the phrasing/wording style each time (direct imperative, polite request, indirect, formal, casual)."
    )
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", enable_thinking=False,
    ).to("cuda")
    with model.disable_adapter():
        out = model.generate(
            input_ids=inputs, max_new_tokens=150,
            temperature=temperature, do_sample=True, top_p=0.95,
        )
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    if "<think>" in text and "</think>" in text:
        text = text.split("</think>", 1)[1].strip()
    return text.strip("\"'' ")

generated_raw = [(steps, generate_command_text(steps)) for steps in candidates]
print(f"Generated {len(generated_raw)} candidate sentences. Example:\n")
print(generated_raw[0][1])


In [ ]:
# ---- 9.5 Round-trip validation: re-extract the generated sentence and require it to match ----
# Reuses run_raw / normalize / step_matches from Section 6 (Cell 12) -- run that cell first if you haven'''t.

def steps_fully_match(pred_steps, gold_steps):
    if not isinstance(pred_steps, list) or len(pred_steps) != len(gold_steps):
        return False
    return all(step_matches(p, g) for p, g in zip(pred_steps, gold_steps))

validated = []
for gold_steps, command_text in generated_raw:
    user_content = f"Parse this robot command:\n{command_text}"
    pred_raw = run_raw(user_content)
    try:
        cleaned = pred_raw.strip()
        if "<think>" in cleaned and "</think>" in cleaned:
            cleaned = cleaned.split("</think>", 1)[1].strip()
        cleaned = cleaned.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        pred_steps = json.loads(cleaned)
    except json.JSONDecodeError:
        pred_steps = None

    if steps_fully_match(pred_steps, gold_steps):
        validated.append({
            "messages": [
                {"role": "user", "content": user_content},
                {"role": "assistant", "content": json.dumps(gold_steps, ensure_ascii=False)},
            ]
        })

print(f"{len(validated)}/{len(generated_raw)} generated examples round-tripped successfully "
      f"({100*len(validated)/len(generated_raw):.1f}%).")
print("The rest were discarded because the phrasing was ambiguous enough that even the model "
      "couldn't recover its own intended labels -- not safe to use as gold-labeled eval data.")


In [ ]:
# ---- 9.6 Save the new test set ----
OUT_PATH = "gpsr_generated_testset.jsonl"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for ex in validated:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"Wrote {len(validated)} examples to {OUT_PATH}")
# Download it (Colab):
from google.colab import files
files.download(OUT_PATH)
